# 18 — Risk Management

## Goal
Calculate the correct **position size** for every trade so that a single loss never exceeds a pre-set fraction of the account.  
This is the most important notebook in the series — a perfect signal with bad sizing is still a losing strategy.

---

## Core Formula

$$\text{position\_size} = \frac{\text{account} \times \text{risk\_pct}}{\text{entry} - \text{sl}}$$

| Variable | Default |
|----------|---------|
| `account` | 10,000 (account size in USD) |
| `risk_pct` | 0.005 (0.5 % per trade) |
| Hard cap | 1 % — never exceed this |

---

## Dollar Risk and Reward

$$\text{dollar\_risk}   = \text{account} \times \text{risk\_pct}$$
$$\text{dollar\_reward} = \text{dollar\_risk} \times 3 \quad (\text{at } 1{:}3 \text{ R:R})$$

---

## Consecutive Loss Drawdown

If we lose $n$ consecutive trades at 0.5 % risk each:

$$\text{drawdown}(n) = 1 - (1 - 0.005)^n$$

| Losing streak | Max drawdown |
|---------------|-------------|
| 5             | ≈ 2.5 % |
| 10            | ≈ 4.9 % |
| 20            | ≈ 9.5 % |

Keeping risk at 0.5 % means a 20-trade losing streak still leaves 90 % of the account intact.

---

## Visualization
Position sizes and dollar risks for each detected zone, plus a drawdown curve.


## 1. Load data and run the detection pipeline

In [ ]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


## 2. Compute trade levels (from NB14) and position sizes

In [ ]:
ACCOUNT   = 10_000.0   # USD
RISK_PCT  = 0.005      # 0.5 % per trade
RISK_HARD_CAP = 0.01   # never exceed 1 %
ATR_BUFFER    = 0.10

def compute_trade_levels(zone):
    avg_atr = atr_arr[zone["bs"]:zone["be"]+1].mean()
    zt      = zone["zone_type"]
    prox    = zone["proximal"]
    dist    = zone["distal"]
    if zt == "demand":
        entry = prox; sl = dist - ATR_BUFFER * avg_atr; risk = entry - sl
        tp    = entry + 3 * risk
    else:
        entry = prox; sl = dist + ATR_BUFFER * avg_atr; risk = sl - entry
        tp    = entry - 3 * risk
    return dict(entry=round(entry,3), sl=round(sl,3), tp=round(tp,3),
                risk=round(risk,3), rr_ok=risk > 0)

for z in zones:
    z.update(compute_trade_levels(z))
    if z["rr_ok"] and z["risk"] > 0:
        effective_risk = min(RISK_PCT, RISK_HARD_CAP)
        dollar_risk    = ACCOUNT * effective_risk
        z["position_size"] = round(dollar_risk / z["risk"], 4)
        z["dollar_risk"]   = round(dollar_risk, 2)
        z["dollar_reward"] = round(dollar_risk * 3, 2)
    else:
        z["position_size"] = 0; z["dollar_risk"] = 0; z["dollar_reward"] = 0
    z["scenario"] = labeled["scenario"].iloc[z["bs"]]

print(f"Account: ${ACCOUNT:,.0f}   Risk per trade: {RISK_PCT*100:.1f}%  "
      f"(max dollar risk: ${ACCOUNT*RISK_PCT:.0f})")
print()
print(f"{'Scenario':<22} {'Type':8} {'Risk pts':>8} {'$ Risk':>7} {'$ Reward':>9} {'Size':>8}")
print("-" * 68)
for z in zones:
    print(f"{z['scenario']:<22} {z['zone_type']:8} {z['risk']:>8.3f} "
          f"${z['dollar_risk']:>6.0f} ${z['dollar_reward']:>8.0f} {z['position_size']:>8.4f}")


## 3. Drawdown curve — consecutive losses

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

n_streak = np.arange(1, 31)
drawdown = 1 - (1 - RISK_PCT) ** n_streak

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Drawdown vs consecutive losses",
    "Position size per zone",
])

fig.add_trace(go.Scatter(
    x=n_streak, y=drawdown * 100,
    mode="lines+markers", line=dict(color="#7c83fd", width=2),
    fill="tozeroy", fillcolor="rgba(124,131,253,0.12)", name="Drawdown %",
), row=1, col=1)
fig.add_hline(y=10, line=dict(color="#ef5350", dash="dash"), row=1, col=1)

tradeable = [z for z in zones if z["rr_ok"] and z["position_size"] > 0]
fig.add_trace(go.Bar(
    x=[z["scenario"] for z in tradeable],
    y=[z["position_size"] for z in tradeable],
    marker_color=[EDGE[z["zone_type"]] for z in tradeable],
    name="Position size",
), row=1, col=2)

fig.update_layout(
    title=f"Risk Management — Account ${ACCOUNT:,.0f}  |  Risk {RISK_PCT*100:.1f}% per trade",
    height=440, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    showlegend=False,
)
for ax in ["xaxis", "xaxis2", "yaxis", "yaxis2"]:
    fig.update_layout(**{ax: dict(gridcolor=GRID, showgrid=True, zeroline=False, linecolor=GRID)})
fig.update_yaxes(title_text="Drawdown %",    row=1, col=1)
fig.update_yaxes(title_text="Position size", row=1, col=2)
fig.update_xaxes(title_text="Consecutive losses", row=1, col=1)
fig.show()
